[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Collections and Documents


## What you will be able to do

Write documents with `insert_one` and `insert_many`, and say what each one hands back. Say where
`_id` comes from when you do not supply one, what an `ObjectId` is made of, and why `insert_one`
changes the dictionary you gave it. Choose your own `_id` when it is worth it. Name the methods that
replaced the pre-4.0 ones, because half the tutorials you will find still use the old names. And
recognize the failure this database is most often accused of: a typo that creates something instead
of complaining.


## The idea

### The problem

Nothing here declares anything. There is no `CREATE TABLE`, so there is no moment at which the
database learns which collections are supposed to exist, and therefore no moment at which it can
tell you that `prodcts` is not one of them.

That is the trade. Writing is frictionless and a misspelling is a silent empty result, forever.

### What a collection is

A named place documents live, inside a database. It comes into being the first time something is
written to it and not before, which means `db.anything` is always a valid expression and almost
never an error.

### What an `_id` is

Every document has one, unique within its collection, and it is the only field MongoDB insists on.
Supply one and it is yours. Leave it out and the **driver**, not the server, generates an
`ObjectId`: twelve bytes of timestamp, a per-process random value, and a counter.

### Why the driver generates it

So that `insert_one` can tell you the `_id` without waiting for a round trip, and so a failed
insert can be retried without creating two documents. The cost is that PyMongo has to put the
`_id` somewhere, and where it puts it is the dictionary you passed in.

### Where this shows up

Every write. The `_id` in particular turns up again in **BSON Types**, where `json.dumps` refuses
it, and in **Beanie Documents**, where it is a field on a Pydantic model.

### What this notebook covers

`insert_one`, `insert_many`, and their results. `ObjectId` and what is inside it. Choosing your own
`_id`. The methods whose names changed in PyMongo 4.0. Then the typo, the old `count()`, a duplicate
`_id`, and the halfway-finished `insert_many`.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

shop.notes.drop()
note = {"title": "first"}                          # no _id anywhere in it
result = shop.notes.insert_one(note)

print("the dict you passed in now has one:", "_id" in note)
print("and it is the id the server acknowledged:", note["_id"] == result.inserted_id)
print("twenty four hex characters of it:", len(str(result.inserted_id)))

print("a collection nobody created answers:", shop.notez.count_documents({}))
print("and it still does not exist:", "notez" in shop.list_collection_names())
client.close()
```

```
the dict you passed in now has one: True
and it is the id the server acknowledged: True
twenty four hex characters of it: 24
a collection nobody created answers: 0
and it still does not exist: False
```

Two surprises in eight lines. The dictionary you handed to `insert_one` was modified: it has an
`_id` now, and it did not before. And `notez`, which is a typo, answered a query rather than
raising, because asking about a collection is not the same as creating one.


## Setup

Eight imports, MongoDB, and the boot cell.

- `pymongo` is the driver, and `ObjectId` is imported from `bson` so this notebook can build one
- `subprocess` and `os` install and start the server, `sys` names this Python, `time` waits for it
- `random` seeds the data the same way every run, with `version` and `PackageNotFoundError`

This is the same boot cell as **Why Documents**, which explains it line by line. It is idempotent:
if a MongoDB is already running on `127.0.0.1:27017` it is left alone, and the seed is skipped when
the collection already holds the right number of documents.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo
from bson import ObjectId

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### insert_one, and what it gives back

The result is small on purpose:


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.notes.drop()

result = shop.notes.insert_one({"title": "a note", "body": "with some text"})

print("type: ", type(result).__name__)
print("acknowledged:", result.acknowledged)
print("the id is an", type(result.inserted_id).__name__, "of", len(result.inserted_id.binary),
      "bytes")


type:  InsertOneResult
acknowledged: True
the id is an ObjectId of 12 bytes


`acknowledged` is `False` only if you asked for an unacknowledged write, which is a thing you can
do and almost never should. Otherwise the server confirmed it.

### The dictionary you passed in

This is the part that catches people, and it is not a bug:


In [3]:
document = {"title": "watch this"}
print("before:", sorted(document))

shop.notes.insert_one(document)
print("after: ", sorted(document))

try:
    shop.notes.insert_one(document)                                 # the same dict, again
except pymongo.errors.DuplicateKeyError:
    print("inserting it a second time is a duplicate key, because it kept the _id")


before: ['title']
after:  ['_id', 'title']
inserting it a second time is a duplicate key, because it kept the _id


PyMongo generates the `_id` in the client and puts it in your dictionary so that you can read it
afterwards. A loop that builds one document, inserts it, mutates a field and inserts it again
therefore writes one document and then fails, rather than writing two.

The fix is a fresh dictionary each time, or `dict(document)` if you must reuse one:


In [4]:
template = {"title": "from a template"}

for number in range(3):
    shop.notes.insert_one({**template, "n": number})                # a new dict every time

print("inserted:", shop.notes.count_documents({"title": "from a template"}))
print("the template is untouched:", sorted(template))


inserted: 3
the template is untouched: ['title']


### What is inside an ObjectId

Twelve bytes, and the first four of them are a timestamp:


In [5]:
made = ObjectId()

print("hex:       ", len(str(made)), "characters, different every time")
print("bytes:     ", len(made.binary))
print("created at:", made.generation_time.tzinfo is not None, "and is a real datetime")
print("ordered:   ", ObjectId() > made, "<- a later one sorts after an earlier one")


hex:        24 characters, different every time
bytes:      12
created at: True and is a real datetime
ordered:    True <- a later one sorts after an earlier one


That ordering is why sorting by `_id` is close to sorting by creation time, and why a query for the
newest documents can use the `_id` index rather than a field of your own.

It is only close, though: the timestamp has one second of resolution and the rest is a counter and
a random value, so two documents made in the same second are ordered arbitrarily between processes.

### Choosing your own _id

Any unique value will do, and a natural key saves an index:


In [6]:
shop.skus.drop()
shop.skus.insert_one({"_id": "LAP-000000", "stock": 3})
shop.skus.insert_one({"_id": "MON-000001", "stock": 11})

print("found by id:", shop.skus.find_one({"_id": "LAP-000000"}))
print("the seeded products did this too:", shop.products.find_one({"_id": 0})["sku"])
print("indexes on skus:", [index["name"] for index in shop.skus.list_indexes()])


found by id: {'_id': 'LAP-000000', 'stock': 3}
the seeded products did this too: LAP-000000
indexes on skus: ['_id_']


One index, the one on `_id` that every collection gets. Had the sku been an ordinary field it would
need an index of its own to be found quickly, which **Indexes** is about.

The cost is that an `_id` cannot be changed. Choose one that is a fact about the thing, not a
decision about it.

### insert_many

One round trip for many documents, and a list of ids back:


In [7]:
shop.notes.drop()
many = shop.notes.insert_many([{"n": number} for number in range(5)])

print("ids:", len(many.inserted_ids), "of them, all", type(many.inserted_ids[0]).__name__)
print("count:", shop.notes.count_documents({}))
print("ordered defaults to True, so a failure stops the rest")


ids: 5 of them, all ObjectId
count: 5
ordered defaults to True, so a failure stops the rest


`ordered=True` is the default: the server applies them in order and stops at the first failure, so
an error halfway leaves the first half written. `ordered=False` keeps going and reports everything
that failed at the end. **Bulk Writes and Transactions** is where that choice starts to matter.

### The names that changed

PyMongo 4.0 removed a lot of methods. These are the current ones:

| What you want | Now | Gone |
|---|---|---|
| how many documents | `count_documents(filter)` | `count()` |
| roughly how many, fast | `estimated_document_count()` | `count()` with no filter |
| write one | `insert_one(doc)` | `insert(doc)` |
| write several | `insert_many(docs)` | `insert([docs])` |
| change one | `update_one(filter, update)` | `update(filter, update)` |
| change several | `update_many(filter, update)` | `update(..., multi=True)` |
| replace one whole document | `replace_one(filter, doc)` | `save(doc)` |
| remove one | `delete_one(filter)` | `remove(filter)` |
| remove several | `delete_many(filter)` | `remove(filter)` |
| change one and read it | `find_one_and_update(...)` | `find_and_modify(...)` |

The default when you meet an old name in a tutorial is to assume the tutorial predates 2021. The
error you get is not "this method was removed", as the Common errors section below shows.

### A loader that can be run twice, finished

The shape worth having: it writes, it can be rerun, and it says what it did.


In [8]:
def load_notes(shop, notes):
    """Insert what is missing and leave what is there, using a key of our own as the _id."""
    wanted = {note["slug"]: note for note in notes}
    present = {row["_id"] for row in shop.notes.find({"_id": {"$in": list(wanted)}}, {"_id": 1})}

    missing = [{**note, "_id": slug} for slug, note in wanted.items() if slug not in present]
    if missing:
        shop.notes.insert_many(missing)                             # one round trip for all of them
    return {"asked": len(wanted), "already there": len(present), "written": len(missing)}


shop.notes.drop()
notes = [{"slug": "alpha", "title": "Alpha"},
         {"slug": "beta", "title": "Beta"},
         {"slug": "gamma", "title": "Gamma"}]

print("first run: ", load_notes(shop, notes))
print("second run:", load_notes(shop, notes))
print("with one more:", load_notes(shop, notes + [{"slug": "delta", "title": "Delta"}]))
print("in the end:", shop.notes.count_documents({}), "documents")


first run:  {'asked': 3, 'already there': 0, 'written': 3}
second run: {'asked': 3, 'already there': 3, 'written': 0}
with one more: {'asked': 4, 'already there': 3, 'written': 1}
in the end: 4 documents


The `slug` became the `_id`, so "have I already written this" is a question the `_id` index answers
and no duplicate can be created even if two copies of this run at once. That is the whole reason to
choose your own `_id`.

**Update Operators** shows the other way to write this, with `upsert=True`, and why it is not
always the same thing.

### Where each part came from

| In the loader | What it relies on | The section that showed it |
|---|---|---|
| `_id` from the slug | an `_id` you choose | Choosing your own _id |
| `{"$in": [...]}` on `_id` | the index every collection has | Choosing your own _id |
| `insert_many(missing)` | one round trip, ids back | insert_many |
| building new dicts | `insert_one` mutating what it is given | The dictionary you passed in |
| `count_documents` | the method that replaced `count` | The names that changed |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/02-collections-and-documents-solutions.ipynb).

**1.** Insert one document and print what `insert_one` handed back.


In [9]:
# your code here


**2.** Show that `insert_one` added an `_id` to the dictionary you gave it.


In [10]:
# your code here


**3.** Insert five documents in one call and count them.


In [11]:
# your code here


**4.** Insert a document with an `_id` you chose, then try the same `_id` again.


In [12]:
# your code here


**5.** Ask a collection that does not exist how many documents it has.


In [13]:
# your code here


**6.** Show that `estimated_document_count` and `count_documents({})` agree here.


In [14]:
# your code here


## Common errors

### No error: the collection a typo invents


In [15]:
print("the real one: ", shop.products.count_documents({}))
print("a typo:       ", shop.prodcts.count_documents({}), "<- no exception, no warning")
print("the typo does not exist yet:", "prodcts" in shop.list_collection_names())

shop.prodcts.insert_one({"oops": True})                             # now it does
print("after one write: ", "prodcts" in shop.list_collection_names())
shop.prodcts.drop()


the real one:  500
a typo:        0 <- no exception, no warning
the typo does not exist yet: False
after one write:  True


Reading from a collection that was never created gives an empty result, and writing to it creates
it. The same is true of databases, so `client.shopp` is a perfectly good expression too.

This is the price of having no schema, and it is worth knowing the two habits that contain it:
name your collections in one module-level place rather than as attributes scattered through the
code, and check `list_collection_names()` when a query returns nothing and you expected rows.


In [16]:
PRODUCTS = "products"                                               # one place, spelled once

here = set(shop.list_collection_names())
print("through the name:", shop[PRODUCTS].count_documents({}))
print("products is really there:", PRODUCTS in here)
print("prodcts never was:      ", "prodcts" in here)


through the name: 500
products is really there: True
prodcts never was:       False


`shop[PRODUCTS]` is also the only way to reach a collection whose name is not a valid Python
identifier, and the only way to reach one whose name starts with an underscore.

### TypeError: 'Collection' object is not callable


In [17]:
shop.products.count()


TypeError: 'Collection' object is not callable. If you meant to call the 'count' method on a 'Collection' object it is failing because no such method exists.

Read that message closely, because it is doing something unusual. `shop.products.count` did not
raise: attribute access on a `Collection` returns a **sub-collection**, so `shop.products.count` is
the collection named `products.count`, which is a perfectly ordinary thing to have. Calling it is
what fails.

PyMongo notices what you probably meant and says so, which is why the message is three sentences
long. The replacement is `count_documents`, and it needs a filter:


In [18]:
print("everything:", shop.products.count_documents({}))
print("one kind:  ", shop.products.count_documents({"kind": "laptop"}))
print("roughly, from metadata, with no filter allowed:", shop.products.estimated_document_count())


everything: 500
one kind:   100
roughly, from metadata, with no filter allowed: 500


`count_documents({})` runs an aggregation over the collection and is exact.
`estimated_document_count()` reads a stored number and is instant, and it can be wrong after an
unclean shutdown. **find and find_one** measures the difference on a large collection.

### pymongo.errors.DuplicateKeyError: E11000 duplicate key error


In [19]:
shop.skus.drop()
shop.skus.insert_one({"_id": "LAP-000000", "stock": 3})
shop.skus.insert_one({"_id": "LAP-000000", "stock": 4})


DuplicateKeyError: E11000 duplicate key error collection: shop.skus index: _id_ dup key: { _id: "LAP-000000" }, full error: {'index': 0, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: shop.skus index: _id_ dup key: { _id: "LAP-000000" }', 'keyPattern': {'_id': 1}, 'keyValue': {'_id': 'LAP-000000'}}

`E11000` is the server's own number for this, and you will see it in logs without the Python class
name attached. It means a unique index was violated, and `_id` always has one.

When "write it if it is not there" is what you meant, say that instead:


In [20]:
outcome = shop.skus.update_one({"_id": "LAP-000000"}, {"$setOnInsert": {"stock": 4}}, upsert=True)
print("matched:", outcome.matched_count, "| upserted:", outcome.upserted_id)
print("the stock is still the original:", shop.skus.find_one({"_id": "LAP-000000"})["stock"])


matched: 1 | upserted: None
the stock is still the original: 3


### No error worth the name: insert_many that stopped halfway


In [21]:
shop.batch.drop()
rows = [{"_id": 1}, {"_id": 2}, {"_id": 2}, {"_id": 3}]             # the third is a duplicate

try:
    shop.batch.insert_many(rows)
except pymongo.errors.BulkWriteError as error:
    print("raised:", type(error).__name__)
    print("but the first two were written:", shop.batch.count_documents({}))
    print("and the fourth was not:", shop.batch.count_documents({"_id": 3}) == 0)


raised: BulkWriteError
but the first two were written: 2
and the fourth was not: True


The exception is real, and so is the half-finished write. `ordered=True` is the default, so the
server stopped at the duplicate and never looked at `{"_id": 3}`.

`ordered=False` changes the arithmetic rather than removing the problem: everything that can be
written is written, and the exception lists what could not.


In [22]:
shop.batch.drop()
try:
    shop.batch.insert_many(rows, ordered=False)
except pymongo.errors.BulkWriteError as error:
    print("written:", shop.batch.count_documents({}), "of 4")
    print("failures reported:", len(error.details["writeErrors"]))
    print("the message alone tells you nothing:", str(error).splitlines()[0])


written: 3 of 4
failures reported: 1
the message alone tells you nothing: batch op errors occurred, full error: {'writeErrors': [{'index': 2, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: shop.batch index: _id_ dup key: { _id: 2 }', 'keyPattern': {'_id': 1}, 'keyValue': {'_id': 2}, 'op': {'_id': 2}}], 'writeConcernErrors': [], 'nInserted': 3, 'nUpserted': 0, 'nMatched': 0, 'nModified': 0, 'nRemoved': 0, 'upserted': []}


That last line is the point. `print(error)` says only that batch op errors occurred; everything
useful is in `error.details["writeErrors"]`, and a program that logs the exception without the
details has thrown away the only information it needed.


In [23]:
for collection in ("notes", "skus", "batch"):
    shop[collection].drop()
client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- A collection exists from the first write, not before. Reading a misspelled one gives an empty
  result and writing to one creates it, which is the cost of having no schema.
- `insert_one` returns an `InsertOneResult` and **adds `_id` to the dictionary you passed in**, so
  reusing that dictionary is a duplicate key error rather than a second document.
- An `ObjectId` is twelve bytes generated by the driver: a one second timestamp, a random value and
  a counter. Sorting by `_id` is close to sorting by creation time, but only close.
- You may choose any unique `_id`. A natural key saves an index and makes a loader idempotent, and
  it can never be changed afterwards.
- `insert_many` is one round trip. `ordered=True`, the default, stops at the first failure and
  leaves the earlier half written.
- `count()` was removed in PyMongo 4.0, and because attribute access returns a sub-collection the
  error is `'Collection' object is not callable` rather than anything about a missing method.
- `BulkWriteError` keeps what actually went wrong in `.details["writeErrors"]`.


## What is next

**BSON Types** is what a document may actually contain: the `set` that cannot be stored, the
`datetime` that loses its microseconds and its timezone, and the `ObjectId` that `json.dumps`
refuses to serialize.


---

&#8592; **Previous:** [Why Documents](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/01-why-documents.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
